# 15 — SinBERT-large (Sinhala-only)

**Named in `model-research.md` §4, never tested.** Rathnayake et al.'s Technique 2 found the best
continuous-fine-tuning order for Sinhala-English to be **BERT → SinBERT → XLM-R**, so SinBERT
carries real signal in the most on-task paper in the library.

**Trained on the `sinhala` track alone.** It is a monolingual Sinhala checkpoint; feeding it Tamil
or romanized text would not be a fair test. Its result is compared against the multilingual
models' `sinhala` cell, not against their pooled score.

**The probe raises a flag worth resolving here:** SinBERT-large tokenizes Sinhala at **4.26
tokens/word**, *worse* than multilingual XLM-R (1.81) and LaBSE (1.74). A monolingual model
fragmenting its own language more than a 100-language model is a real finding either way — either
its tokenizer was trained on too little data, or fertility is again failing to predict quality.

**Tokenizer fertility** (tokens per word, measured on 300 dev tickets per language):

| english | sinhala | singlish | tamil | tamilish |
|---|---|---|---|---|
| 5.09 | 4.26 | 6.65 | 6.95 | 7.2 |

Its 0% UNK is not evidence of coverage — RoBERTa byte-level BPE structurally cannot emit UNK.

---

**Protocol.** Fit on `train` (8,500 ids), select the epoch on **dev** `negative_f1`. Test is not
opened here — only the winner of `30_encoder_leaderboard.ipynb` is refit on train+dev and scored
on test. Training code is `swiftbench.train_encoder`, shared with the other five notebooks so the
numbers land in one table.

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, metrics, splits, train_encoder as te

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
print("split sha:", splits.sha(), "| device:", te.device())

## Baseline to beat

The classical champion from `10_final_test_eval.ipynb`, on the same dev split. An encoder that
does not clear this is not worth its serving cost.

In [ ]:
CLASSICAL_DEV = 0.6144      # tfidf-svm / class_weight / multi, pooled dev negative_f1
CLASSICAL_TEST = 0.4572     # same model, pooled test

runs = sb.results.load_all("dev")
if not runs.empty and "family" in runs.columns:
    done = runs[(runs.task == "sentiment") & (runs.family == "encoder")]
    if not done.empty:
        display(done[["model", "arm", "eval_lang", "headline", "best_epoch", "train_seconds"]]
                .sort_values("headline", ascending=False))
print(f"classical dev  negative_f1 {CLASSICAL_DEV:.4f}")
print(f"classical test negative_f1 {CLASSICAL_TEST:.4f}")

## Fine-tune

`SMOKE = True` runs a 1,200-row sanity pass in about a minute. Set it to `False` for the real
run (this one is cheap — Sinhala track only).

In [ ]:
SMOKE = True          # <- set False for the real run

EPOCHS = 3
BATCH_SIZE = 32
LR = 2e-5
ARM = "class_weight"  # weighted loss; `ros` and `none` are the other two arms

run = te.run(
    task="sentiment",
    model="sinbert-large",
    train_langs=["sinhala"], eval_lang="sinhala",
    arm=ARM,
    portion="dev",
    fit_portion="train",
    epochs=1 if SMOKE else EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    subsample=1200 if SMOKE else None,
    author=AUTHOR,
    save=not SMOKE,
)
run.scores

In [ ]:
display(run.history)
print(f"selected epoch {run.scores['best_epoch']} of {run.scores['epochs']}  "
      f"({run.scores['train_seconds']/60:.1f} min on {run.scores['device']})")

## Where it fails

Selection metric alone hides the operating point. At 95%+ Neutral, a model can post a healthy
Negative-F1 while its precision makes escalation unusable — the bake-off's encoders all sat at
0.12-0.20 precision against 0.77-0.90 recall.

In [ ]:
ev = run.eval_frame.copy()
ev["pred"] = run.predictions
ev["p_negative"] = run.scores_positive

print("confusion (rows = truth)")
labels, cm = metrics.confusion(ev.sentiment, ev.pred, "sentiment")
display(pd.DataFrame(cm, index=labels, columns=labels))

print(f"\nprecision {run.scores['negative_precision']:.4f}   "
      f"recall {run.scores['negative_recall']:.4f}   "
      f"negative_f1 {run.scores['headline']:.4f}")

In [ ]:
# Per-language breakdown — this model only sees sinhala, shown for consistency.
rows = []
for lang in ['sinhala']:
    m = (ev.language == lang)
    if not m.any():
        continue
    s = metrics.score(ev.sentiment[m], ev.pred[m], "sentiment")
    rows.append({"language": lang, **{k: v for k, v in s.items() if not isinstance(v, str)}})
per_lang = pd.DataFrame(rows)
display(per_lang[["language", "negative_f1", "negative_precision", "negative_recall",
                  "accuracy", "n_negative_true"]])

In [ ]:
# False negatives -- angry customers routed to an auto-reply. These are the expensive errors.
missed = ev[(ev.sentiment == "Negative") & (ev.pred != "Negative")]
print(f"{len(missed)} missed Negative rows of {(ev.sentiment == 'Negative').sum()}")
display(missed.sort_values("p_negative", ascending=False)[["language", "text", "p_negative"]].head(12))

## Verdict

Fill this in after the real run.

- Beats classical dev (0.6144) — yes / no, and by how much against the CI width of ~0.15.
- Where it wins and loses per language, especially the `sinhala` cell.
- Whether precision is high enough for escalation, or whether it needs the lexicon layer from
  `20_technique_lexicon_correction.ipynb`.

Then record it in `30_encoder_leaderboard.ipynb`.